# 03 — Feature engineering

Use domain knowledge to propose features, then verify them against local validation. Apply deterministic row-wise transforms consistently to train and test. Any transform that learns from data (scaling, imputation, category/target encoding, feature selection) must be fit inside each CV fold to avoid leakage.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data/raw")
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
print("Candidate columns:", train.columns.tolist())

Example pattern for a competition that has a date column: parse the same column in both splits and derive simple calendar features. Adapt the column name and timezone/format handling to the data dictionary. Do not add a feature simply because it improves one validation run; check stability across folds.

In [ ]:
DATE_COLUMN = "date"  # TODO: replace, or skip this example if there is no date
if DATE_COLUMN in train.columns and DATE_COLUMN in test.columns:
    for frame in (train, test):
        date = pd.to_datetime(frame[DATE_COLUMN], errors="coerce")
        frame[f"{DATE_COLUMN}_year"] = date.dt.year
        frame[f"{DATE_COLUMN}_month"] = date.dt.month
        frame[f"{DATE_COLUMN}_dayofweek"] = date.dt.dayofweek
    print("Added calendar features to both splits.")
else:
    print(f"No matching {DATE_COLUMN!r} column in both splits; no features added.")

Move reusable transforms into your project code and test train/test schema alignment. For learned transforms, add a scikit-learn transformer to the pipeline in `src/kaggle_template/train.py` rather than fitting it before cross-validation.